# Student Notebook - Project Option B
## FMCW Range-Gated Physiological Sensing

Complete every `TODO` code cell and replace every **[Write your response here.]** prompt. Filtering helpers are provided; the range FFT, physical axes, target selection, phase recovery, and interpretation are your responsibility.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.signal import butter, sosfiltfilt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.titlesize": 12})

def find_data_file(filename):
    """Find a distributed data file from common notebook locations."""
    candidates = [
        Path(filename),
        Path("data") / filename,
        Path("../data") / filename,
        Path("../../data") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {filename}. Tried: {candidates}")

In [ ]:
def bandpass_zero_phase(signal, low_hz, high_hz, fs_hz, order=4):
    """Butterworth bandpass applied forward and backward for zero phase shift."""
    sos = butter(order, [low_hz, high_hz], btype="bandpass", fs=fs_hz, output="sos")
    return sosfiltfilt(sos, signal)


def one_sided_amplitude_spectrum(signal, fs_hz):
    """Hann-windowed one-sided amplitude spectrum."""
    signal = np.asarray(signal)
    window = np.hanning(signal.size)
    frequency_hz = np.fft.rfftfreq(signal.size, 1/fs_hz)
    amplitude = 2*np.abs(np.fft.rfft((signal-signal.mean())*window))/window.sum()
    return frequency_hz, amplitude

## 1. Load and inspect the FMCW data

In [ ]:
data_path = find_data_file("./data/fmcw_physiological_data.npz")
data = np.load(data_path)
print("Available arrays/metadata:", data.files)

# TODO: load I and Q and form z = I + 1j*Q.
# TODO: extract all radar/acquisition metadata into clearly named variables.
# TODO: print the I/Q shapes and identify the fast- and slow-time axes.

# for key in data.files:
#     arr = data[key]
#     if arr.ndim == 0:
#         print(f"{key:24s} scalar = {arr}")
#     else:
#         print(f"{key:24s} shape={arr.shape} dtype={arr.dtype}")

data_path = find_data_file("./data/fmcw_physiological_data.npz")
data = np.load(data_path)
print("Available arrays/metadata:", data.files)

#TODO 1:
I = data["I"]
Q = data["Q"]
z = I + 1j*Q

#TODO 2:
f_center_hz       = float(data["fc_hz"])
bandwidth_hz      = float(data["bandwidth_hz"])
chirp_duration_s  = float(data["chirp_duration_s"])
chirp_slope       = float(data["chirp_slope_hz_per_s"])
fs_fast_hz        = float(data["fs_fast_hz"])
fs_slow_hz        = float(data["fs_slow_hz"])
n_fast            = int(data["num_fast_samples"])
n_chirps          = int(data["num_chirps"])
record_duration_s = float(data["record_duration_s"])

c            = 3e8
wavelength_m = c/f_center_hz
range_res_m  = c/(2*bandwidth_hz)
max_range_m  = c*(fs_fast_hz/2)/(2*chirp_slope)

#TODO 3:
print(f"I {I.shape}, Q {Q.shape}, z dtype {z.dtype}")
print(f"axis 0 slow time: {z.shape[0]} chirps @ {fs_slow_hz:.1f} Hz -> {record_duration_s:.1f} s")
print(f"axis 1 fast time: {z.shape[1]} samples @ {fs_fast_hz/1e3:.1f} kHz -> {chirp_duration_s*1e3:.2f} ms")
print(f"fc {f_center_hz/1e9:.1f} GHz, lambda {wavelength_m*1000:.2f} mm, B {bandwidth_hz/1e9:.2f} GHz")
print(f"slope {chirp_slope/1e12:.2f} THz/s, range res {range_res_m*100:.1f} cm, max range {max_range_m:.1f} m")

# consistency guards
assert z.shape == (n_chirps, n_fast)
assert abs(n_fast/fs_fast_hz - chirp_duration_s)/chirp_duration_s < 0.05
assert abs(bandwidth_hz/chirp_duration_s - chirp_slope)/chirp_slope < 0.05
assert abs(n_chirps/fs_slow_hz - record_duration_s)/record_duration_s < 0.05

unwrap_limit_mm_hz = wavelength_m*fs_slow_hz/(4*np.pi)*1000
print(f"unwrap ceiling: (pk-pk mm) x (motion Hz) < {unwrap_limit_mm_hz:.1f}")

**Checkpoint 1 - Data dimensions**

What does one row represent? What does one column represent? Why is an FMCW record more structured than a CW I/Q time series?

**[Write your response here.]**

## 2. Inspect one raw dechirped chirp

In [ ]:
# TODO: construct the fast-time axis.
t_fast_s = np.arange(n_fast)/fs_fast_hz

# TODO: plot I and Q for one chirp versus fast time.
chirp_idx = 0

fig, ax = plt.subplots()
ax.plot(t_fast_s*1e6, I[chirp_idx], label="I", linewidth=1.2)
ax.plot(t_fast_s*1e6, Q[chirp_idx], label="Q", linewidth=1.2)
ax.set_xlabel("Fast time (us)")
ax.set_ylabel("Amplitude")
ax.set_title(f"Dechirped chirp {chirp_idx} — I and Q vs fast time")
ax.legend()
plt.tight_layout()
plt.show()

**Checkpoint 2 - Before the range FFT**

Why is the raw dechirped chirp not yet a range profile? What physical quantity is encoded in its beat frequencies?

**[Write your response here.]**

## 3. Range processing

In [ ]:
# TODO: apply a Hann window along the fast-time axis.
w_fast = np.hanning(n_fast)
z_win = z * w_fast[np.newaxis, :]

# TODO: calculate the FFT along axis 1 and retain positive range bins.
n_pos = n_fast//2
range_fft = np.fft.fft(z_win, axis=1)[:, :n_pos]     # (6000, 64) complex

# TODO: build beat-frequency and range axes using the supplied metadata.
beat_freq_hz = np.fft.fftfreq(n_fast, 1/fs_fast_hz)[:n_pos]
range_axis_m = beat_freq_hz*c/(2*chirp_slope)

# TODO: calculate theoretical range resolution c/(2B).
range_res_m = c/(2*bandwidth_hz)

print(f"range_fft {range_fft.shape}, {n_pos} positive bins")
print(f"bin spacing {range_axis_m[1]:.3f} m, max unambiguous {range_axis_m[-1]:.2f} m")
print(f"theoretical range resolution {range_res_m*100:.1f} cm")

In [ ]:
# TODO: average range-FFT magnitude or power across chirps.
mean_profile = np.mean(np.abs(range_fft), axis=0)    # magnitude first, then average

# TODO: plot the mean range profile and identify the physiological target bin.
profile_db = 20*np.log10(mean_profile/mean_profile.max())

# Do not average the complex FFT before taking magnitude.
leakage_skip = 3
search = mean_profile.copy()
search[:leakage_skip] = 0.0
target_bin = int(np.argmax(search))
target_range_m = range_axis_m[target_bin]

fig, ax = plt.subplots()
ax.plot(range_axis_m, profile_db, linewidth=1.2)
ax.axvline(target_range_m, color="tab:red", linestyle=":",
           label=f"target bin {target_bin} ({target_range_m:.2f} m)")
ax.set_xlabel("Range (m)")
ax.set_ylabel("Relative magnitude (dB)")
ax.set_title("Mean range profile")
ax.legend()
plt.tight_layout()
plt.show()

print(f"target bin {target_bin} at {target_range_m:.2f} m, "
      f"{profile_db[target_bin]:.1f} dB rel. peak")
for b in range(max(0, target_bin-2), min(n_pos, target_bin+3)):
    print(f"  bin {b:2d}  {range_axis_m[b]:5.2f} m  {profile_db[b]:6.1f} dB")

**Checkpoint 3 - Range profile**

Why could complex averaging suppress the physiological target? Report the selected target bin, estimated range, and theoretical range resolution.

**[Write your response here.]**

## 4. Range-time map

In [ ]:
# TODO: create a relative-dB range-versus-slow-time magnitude heatmap.
# Mark the selected physiological target range.
mag = np.abs(range_fft)                      # (6000, 64)
mag_db = 20*np.log10(mag/mag.max() + 1e-12)

t_slow_s = np.arange(z.shape[0])/fs_slow_hz

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.pcolormesh(
    t_slow_s, range_axis_m, mag_db.T,
    shading="nearest", cmap="viridis", vmin=-60, vmax=0,
)
ax.axhline(target_range_m, color="tab:red", linestyle=":", linewidth=1.5,
           label=f"target bin {target_bin} ({target_range_m:.2f} m)")
ax.set_xlabel("Slow time (s)")
ax.set_ylabel("Range (m)")
ax.set_title("Range vs slow time, relative magnitude")
ax.legend(loc="upper right")
fig.colorbar(im, ax=ax, label="Relative magnitude (dB)")
plt.tight_layout()
plt.show()

peak_bins = np.argmax(mag, axis=1)
print(f"per-chirp peak bin: min {peak_bins.min()}, max {peak_bins.max()}, "
      f"unique {np.unique(peak_bins)}")

**Checkpoint 4 - Coarse range versus small motion**

Does the target visibly move across range bins? Explain why phase can recover motion much smaller than the range resolution.

**[Write your response here.]**

## 5. Extract the complex target bin and recover displacement

In [ ]:
# TODO: extract target_signal = range_fft[:, target_bin].
target_signal = range_fft[:, target_bin]            # (6000,) complex

# TODO: plot its I/Q constellation.
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(target_signal.real, target_signal.imag, "-", linewidth=0.6, alpha=0.7)
ax.set_aspect("equal", "datalim")
ax.set_xlabel("I")
ax.set_ylabel("Q")
ax.set_title(f"I/Q constellation — bin {target_bin} ({target_range_m:.2f} m)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# TODO: calculate and separately plot wrapped and unwrapped phase.
phase_wrapped   = np.angle(target_signal)
phase_unwrapped = np.unwrap(phase_wrapped)

fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axs[0].plot(t_slow_s, phase_wrapped, linewidth=0.8)
axs[0].set_ylabel("Wrapped phase (rad)")
axs[0].grid(True, alpha=0.3)
axs[1].plot(t_slow_s, phase_unwrapped, linewidth=0.8, color="tab:orange")
axs[1].set_xlabel("Slow time (s)")
axs[1].set_ylabel("Unwrapped phase (rad)")
axs[1].grid(True, alpha=0.3)
fig.suptitle(f"Phase at bin {target_bin}")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# TODO: convert phase to displacement using lambda/(4*pi).
displacement_mm = phase_unwrapped*wavelength_m/(4*np.pi)*1000
displacement_mm -= displacement_mm.mean()

fig, ax = plt.subplots()
ax.plot(t_slow_s, displacement_mm, linewidth=0.8)
ax.set_xlabel("Slow time (s)")
ax.set_ylabel("Displacement (mm)")
ax.set_title(f"Recovered displacement — bin {target_bin}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

step = np.abs(np.diff(phase_unwrapped))
print(f"phase step max {step.max():.3f} rad, p99 {np.percentile(step, 99):.3f} rad")
print(f"arc span {np.ptp(phase_unwrapped)*180/np.pi:.1f} deg, "
      f"{np.ptp(phase_unwrapped)/(2*np.pi):.2f} revolutions")
print(f"displacement peak-to-peak {np.ptp(displacement_mm):.3f} mm")

**Checkpoint 5 - Preserve complex phase**

Why would using only `abs(range_fft)` make physiological displacement recovery impossible? Clearly distinguish wrapped phase, unwrapped phase, and displacement.

**[Write your response here.]**

## 6. Estimate physiological rates

In [ ]:
from scipy.signal import detrend
disp_dt = detrend(displacement_mm, type="linear")
print(f"raw pk-pk {np.ptp(displacement_mm):.2f} mm, detrended {np.ptp(disp_dt):.2f} mm")

# TODO: use one_sided_amplitude_spectrum on recovered displacement.
f_disp, a_disp = one_sided_amplitude_spectrum(displacement_mm, fs_slow_hz)

# TODO: search 0.1-0.5 Hz for respiration and 0.8-2.0 Hz for heart rate.
def peak_in_band(freq, amp, f_lo, f_hi):
    band = (freq >= f_lo) & (freq <= f_hi)
    idx = np.flatnonzero(band)[np.argmax(amp[band])]
    return freq[idx], amp[idx]

resp_hz, resp_amp = peak_in_band(f_disp, a_disp, 0.1, 0.5)
hr_hz,   hr_amp   = peak_in_band(f_disp, a_disp, 0.8, 2.0)

# TODO: report both estimates in Hz and cycles/min.
print(f"respiration {resp_hz:.3f} Hz = {resp_hz*60:.1f} breaths/min, "
      f"amplitude {resp_amp:.3f} mm")
print(f"heart rate  {hr_hz:.3f} Hz = {hr_hz*60:.1f} bpm, "
      f"amplitude {hr_amp:.4f} mm")
print(f"frequency resolution {f_disp[1]:.4f} Hz")

# TODO: plot the spectrum through 10 Hz and identify cardiac harmonics.
fig, axs = plt.subplots(2, 1, figsize=(10, 7))

axs[0].plot(f_disp, a_disp, linewidth=1.0)
axs[0].axvline(resp_hz, color="tab:green", linestyle=":",
               label=f"resp {resp_hz:.3f} Hz")
axs[0].axvline(hr_hz, color="tab:red", linestyle=":",
               label=f"HR {hr_hz:.3f} Hz")
axs[0].set_xlim(0, 3)
axs[0].set_xlabel("Frequency (Hz)")
axs[0].set_ylabel("Amplitude (mm)")
axs[0].set_title("Displacement spectrum, 0-3 Hz")
axs[0].legend()
axs[0].grid(True, alpha=0.3)

axs[1].semilogy(f_disp, a_disp, linewidth=1.0)
for k in range(1, 9):
    fk = hr_hz*k
    if fk <= 10:
        axs[1].axvline(fk, color="tab:red", linestyle=":", alpha=0.6)
        axs[1].annotate(f"{k}x", (fk, a_disp.max()), fontsize=8,
                        ha="center", va="top")
axs[1].set_xlim(0, 10)
axs[1].set_xlabel("Frequency (Hz)")
axs[1].set_ylabel("Amplitude (mm, log)")
axs[1].set_title(f"Spectrum through 10 Hz with cardiac harmonics of {hr_hz:.3f} Hz")
axs[1].grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.show()

# harmonic amplitudes at the marked lines
for k in range(1, 9):
    fk = hr_hz*k
    if fk <= 10:
        i = int(np.argmin(np.abs(f_disp - fk)))
        print(f"  harmonic {k}: {f_disp[i]:5.3f} Hz, {a_disp[i]:.5f} mm")


**Checkpoint 6 - Spectrum interpretation**

Report breathing and heart rates. Why does the pulse-like cardiac motion produce harmonics rather than only one spectral line?

**[Write your response here.]**

## 7. Filter respiration and cardiac motion

In [ ]:
# TODO: use bandpass_zero_phase to calculate:
#   respiration: 0.1-0.5 Hz
#   heart-rate signal: 0.8-2.0 Hz
#   cardiac waveform: 0.8-8.0 Hz
resp_band    = bandpass_zero_phase(displacement_mm, 0.1, 0.5, fs_slow_hz)
cardiac_narrow = bandpass_zero_phase(displacement_mm, 0.8, 2.0, fs_slow_hz)
cardiac_wide   = bandpass_zero_phase(displacement_mm, 0.8, 8.0, fs_slow_hz)

# TODO: plot all three with appropriate independent vertical scales.
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axs[0].plot(t_slow_s, resp_band, linewidth=1.0)
axs[0].set_ylabel("Respiration (mm)")
axs[0].set_title("0.1-0.5 Hz")
axs[0].grid(True, alpha=0.3)

axs[1].plot(t_slow_s, cardiac_narrow, linewidth=1.0, color="tab:orange")
axs[1].set_ylabel("Heart rate (mm)")
axs[1].set_title("0.8-2.0 Hz")
axs[1].grid(True, alpha=0.3)

axs[2].plot(t_slow_s, cardiac_wide, linewidth=1.0, color="tab:green")
axs[2].set_xlabel("Slow time (s)")
axs[2].set_ylabel("Cardiac waveform (mm)")
axs[2].set_title("0.8-8.0 Hz")
axs[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"respiration pk-pk   {np.ptp(resp_band):.3f} mm")
print(f"cardiac narrow pk-pk {np.ptp(cardiac_narrow):.4f} mm")
print(f"cardiac wide pk-pk   {np.ptp(cardiac_wide):.4f} mm")

# TODO: directly compare the two cardiac filters over a 5-second interval.
t0, t1 = 20.0, 25.0
seg = (t_slow_s >= t0) & (t_slow_s <= t1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_slow_s[seg], cardiac_narrow[seg], linewidth=1.4, label="0.8-2.0 Hz")
ax.plot(t_slow_s[seg], cardiac_wide[seg], linewidth=1.0, label="0.8-8.0 Hz")
ax.set_xlabel("Slow time (s)")
ax.set_ylabel("Displacement (mm)")
ax.set_title(f"Cardiac filters compared, {t0:.0f}-{t1:.0f} s")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

beats = (t1-t0)*hr_hz
print(f"{beats:.1f} beats expected in the {t1-t0:.0f} s window")

## Final engineering interpretation

Address all of the following:

1. What processing step gives range?
2. What processing step gives small displacement?
3. Why must the selected range bin remain complex?
4. Why do the narrow- and wide-band heartbeat signals have different shapes?
5. What does FMCW add relative to a single-frequency CW physiological radar?

**[Write your response here.]**

Your presentation should summarize the processing chain, target selection, most important plots, rate estimates, and engineering conclusions. Do not read every notebook answer verbatim.